In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="this year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F57006EC50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5701D6710>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out 

In [7]:
response = model_with_structure.invoke("Details of movie dhurandar")
response

Movie(title='Dhurandar', year=2023, director='Sanjay Leela Bhansali', rating=8.5)

Message output alongside parsed structure

In [9]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="this year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Details of movie dhurandar")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "dhurandar". First, I need to check if I have access to the right tools. The provided tool is the Movie function, which requires title, year, director, and rating. I need to find out if "dhurandar" is a real movie and gather those details.\n\nWait, "dhurandar" might be a typo. Let me think. There\'s a famous Marathi film called "Dhurandhar" directed by Raj N. Sippy, released in 2003. The user might have misspelled it. The correct title is "Dhurandhar". The director is Raj N. Sippy, and the year is 2003. I need to confirm the rating. Let me check my sources. The movie has a rating of 8.5/10 on IMDb. So the parameters would be title: Dhurandhar, year: 2003, director: Raj N. Sippy, rating: 8.5. \n\nI should call the Movie function with these details. Since the user used the misspelled title, I should correct it in the response. The required fields are all present, so

Nested Structure

In [13]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class Moviedetails(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="this year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(Moviedetails)
response = model_with_structure.invoke("Details of movie dhurandhar")
response


Moviedetails(title='Dhurandhar', year=2016, director='Vinayak Bangar', rating=7.8, cast=[Actor(name='Srushti Sinha', role='Dhurandhar'), Actor(name='Sachin Pilgaonkar', role='Supporting Role')], genres=['Drama', 'Romance'], budget=None)

In [14]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

TypedDict

In [1]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContacInfo(TypedDict):
    """Contact information for a person"""
    name:str #the name of the person
    email:str #the email of the person
    phone:str #the phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContacInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: mAYURESH MANDALIK , mayur@gmail.com, 98978786766"}]
})

result["structured_response"]

{'name': 'Mayuresh Mandalik',
 'email': 'mayur@gmail.com',
 'phone': '98978786766'}

DataClasses


In [2]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContacInfo:
    """Contact information for a person"""
    name:str #the name of the person
    email:str #the email of the person
    phone:str #the phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContacInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: mAYURESH MANDALIK , mayur@gmail.com, 98978786766"}]
})

result["structured_response"]

ContacInfo(name='Mayuresh Mandalik', email='mayur@gmail.com', phone='98978786766')